# CareTrace — Demo Scenarios (all 5)

**DATASCI 290 — Neurosymbolic AI, Spring 2026**

All 5 evaluation scenarios with **full per-turn state → rule → decision trace**:

| # | Type | Expected |
|---|---|---|
| 1 | Base — Home management | HOME_MANAGEMENT |
| 2 | Base — ER now | ER_NOW |
| 3 | Base — Urgent same-day | URGENT_SAME_DAY |
| 4 | **Extended — Medication conflict** (ibuprofen age gate, infant <6 mo) | HOME_MANAGEMENT + med flag |
| 5 | **Extended — Local context** (viral context captured, never overrides gate) | HOME_MANAGEMENT |

Run with: `CARETRACE_MOCK_LLM=1 CARETRACE_SKIP_NEO4J=1`

In [71]:
from __future__ import annotations
import os
os.environ.setdefault('CARETRACE_MOCK_LLM', '0')
os.environ.setdefault('CARETRACE_SKIP_NEO4J', '0')
os.environ.setdefault('CARETRACE_USE_LAG', '1')
from caretrace.orchestration.graph import run_turn
from caretrace.state import default_case

print(f"Ready with switch - >  mock_llm={os.environ.get('CARETRACE_MOCK_LLM')}  skip_neo4j={os.environ.get('CARETRACE_SKIP_NEO4J')}")

Ready with switch - >  mock_llm=0  skip_neo4j=0


In [72]:
def run_scenario(title: str, expected: str, turns: list[str], verbose: bool = True) -> dict:
    """
    Run a multi-turn scenario with per-turn state trace.
    Shows: input → structured state → missing fields → KG → rules fired → decision
    """
    print(f'\n{"="*68}')
    print(f'  {title}')
    print(f'  Expected disposition: {expected}')
    print(f'{"="*68}')

    state: dict = {'messages': [], 'case': default_case(), 'kg_annotations': [], 'turn': 0}

    for i, text in enumerate(turns, 1):
        state = dict(state)
        state['raw_user_text'] = text
        msgs = list(state.get('messages') or [])
        msgs.append({'role': 'user', 'content': text})
        state['messages'] = msgs
        state = run_turn(state)

        case = state.get('case', {})
        dec  = state.get('decision', {})
        kg   = state.get('kg_annotations', [])
        active = {k: v for k, v in case.items() if v not in (None, [], False, 'unknown')}

        if verbose:
            print(f'\n  Turn {i}:')
            print(f'  ┌─ INPUT  : {text}')
            print(f'  ├─ STATE  : {active}')
            print(f'  ├─ KG     : {len(kg)} concepts (Neo4j {"offline" if not kg else "live"}) -> Concepts: {kg}')
            print(f'  ├─ MISSING: {dec.get("missing_required", [])}')
            print(f'  ├─ RULES  : {dec.get("rule_ids", [])}')
            print(f'  ├─ MFLAGS : {dec.get("med_flags", [])}')
            disp = dec.get('disposition')
            print(f'  └─ \033[1mDECISION : {disp}\033[0m')

    reply = state.get('assistant_reply', '')
    print(f'\n--- CareTrace Response ---')
    print(reply)

    actual = state.get('decision', {}).get('disposition')
    result = '✓ PASS' if actual == expected else f'✗ FAIL (got {actual})'
    print(f'\n{result}')
    return state

---
## Scenario 1 — Home Management

6-year-old with moderate fever (101.8°F), tired but responsive, sipping fluids, on amoxicillin.

**Expected:** `HOME_MANAGEMENT` — child is stable; antibiotic interaction flag surfaced.

In [73]:
s1 = run_scenario(
    title='Scenario 1 — Home Management (6-year-old, moderate fever)',
    expected='HOME_MANAGEMENT',
    turns=[
        'My 6-year-old has a fever, threw up once, and looks really wiped out.',
        "Temp is 101.8. He's tired but answers me. No breathing issues. He's sipping water, not much though. He's been on medication for a recent ear infection.",
        "He's on amoxicillin. Last dose was earlier tonight. Just vomited once. He peed earlier this evening.",
    ]
)


  Scenario 1 — Home Management (6-year-old, moderate fever)
  Expected disposition: HOME_MANAGEMENT
In Format_lag_context
--------------------------------
Extnded Context:  === SYMBOLIC TRIAGE CONTEXT ===

PATIENT STATE (structured from caregiver reports):
- Age: 6 years
- Temperature: not yet reported
- Alertness: unknown
- Breathing: unknown
- Fluid intake: unknown
- Urination last 8h: unknown
- Vomiting: once
- Current medications: none reported

KNOWLEDGE GRAPH EVIDENCE:
KG concepts annotated from caregiver text:
  - Finding of vomiting [SNOMED 300359004]  ← matched phrase: "vomiting"
  - Vomiting [SNOMED 422400008]  ← matched phrase: "vomiting"
  - Fever [SNOMED 386661006]  ← matched phrase: "fever"

TRIAGE RULES EVALUATED (all rules in scope):
✗ NOT FIRED: R_ER_ALERTNESS  (Altered alertness)
✗ NOT FIRED: R_ER_BREATHING  (Breathing distress)
✗ NOT FIRED: R_ER_DEHYDRATION_SEVERE  (Severe dehydration)
✗ NOT FIRED: R_ER_NO_FLUID_NO_URINE  (No fluid intake and no urine output)
✗ NOT 

---
## Scenario 2 — ER Now

6-year-old, 103.5°F, **barely responding**, not drinking, no urine since afternoon.
Local viral context mentioned — must NOT override hard safety gates.

**Expected:** `ER_NOW` — `R_ER_ALERTNESS` + `R_ER_DEHYDRATION_SEVERE` both fire.

In [74]:
s2 = run_scenario(
    title='Scenario 2 — ER Now (6-year-old, altered alertness + dehydration)',
    expected='ER_NOW',
    turns=[
        "My 6-year-old has a fever, threw up, and looks really wiped out. I'm worried.",
        "Temp is 103.5. He's barely responding, just lying there. He doesn't want to drink. No trouble breathing. Also, there's been a stomach virus going around his school this week.",
        "I don't think he's peed since this afternoon.",
    ]
)


  Scenario 2 — ER Now (6-year-old, altered alertness + dehydration)
  Expected disposition: ER_NOW
In Format_lag_context
--------------------------------
Extnded Context:  === SYMBOLIC TRIAGE CONTEXT ===

PATIENT STATE (structured from caregiver reports):
- Age: 6 years
- Temperature: not yet reported
- Alertness: unknown
- Breathing: unknown
- Fluid intake: unknown
- Urination last 8h: unknown
- Vomiting: once
- Current medications: none reported

KNOWLEDGE GRAPH EVIDENCE:
KG concepts annotated from caregiver text:
  - Finding of vomiting [SNOMED 300359004]  ← matched phrase: "vomiting"
  - Vomiting [SNOMED 422400008]  ← matched phrase: "vomiting"
  - Fever [SNOMED 386661006]  ← matched phrase: "fever"

TRIAGE RULES EVALUATED (all rules in scope):
✗ NOT FIRED: R_ER_ALERTNESS  (Altered alertness)
✗ NOT FIRED: R_ER_BREATHING  (Breathing distress)
✗ NOT FIRED: R_ER_DEHYDRATION_SEVERE  (Severe dehydration)
✗ NOT FIRED: R_ER_NO_FLUID_NO_URINE  (No fluid intake and no urine output)
✗ NOT F

---
## Scenario 3 — Urgent Same-Day

5-year-old, 102.5°F, **repeated vomiting (×4) + poor fluid intake**.

**Expected:** `URGENT_SAME_DAY` — `R_URGENT_REPEATED_VOMIT_POOR_FLUID` fires.

In [ ]:
s3 = run_scenario(
    title='Scenario 3 — Urgent Same-Day (repeated vomiting + poor intake)',
    expected='URGENT_SAME_DAY',
    turns=[
        '5 year old fever and vomiting',
        "102.5 fever, breathing fine, answers questions, he keeps throwing up and won't drink much, he peed an hour ago",
        'he vomited 4 times in the last 2 hours and only sips',
    ]
)

---
## Scenario 4 — Medication Conflict (Extended)

**5-month-old** baby, 101.5°F, responsive, sipping formula.

Demonstrates the **CPG ibuprofen age gate**:
- Child is **under 6 months** → `cpg_ibuprofen_under_6mo_requires_clinician` med flag fires
- The system surfaces an explicit medication safety warning grounded in the CPG
- The baseline has no equivalent age gate and may suggest ibuprofen freely

**Expected:** `HOME_MANAGEMENT` with med flag `cpg_ibuprofen_under_6mo_requires_clinician`.

In [75]:
s4 = run_scenario(
    title='Scenario 4 — Medication Conflict (5-month-old, ibuprofen age gate)',
    expected='HOME_MANAGEMENT',
    turns=[
        'My 5-month-old baby has a temperature of 101.5 F. She is alert and answers me when I talk to her.',
        'No breathing issues. She is sipping some formula.',
        'She peed an hour ago.',
    ]
)



  Scenario 4 — Medication Conflict (5-month-old, ibuprofen age gate)
  Expected disposition: HOME_MANAGEMENT
In Format_lag_context
--------------------------------
Extnded Context:  === SYMBOLIC TRIAGE CONTEXT ===

PATIENT STATE (structured from caregiver reports):
- Age: 5 months
- Temperature: 101.5°F  [classified: non_extreme (<103°F)]
- Alertness: normal (awake, responsive)
- Breathing: unknown
- Fluid intake: unknown
- Urination last 8h: unknown
- Vomiting: unknown
- Current medications: none reported

KNOWLEDGE GRAPH EVIDENCE:
KG concepts annotated from caregiver text:
  - Fever [SNOMED 386661006]  ← matched phrase: "fever"

TRIAGE RULES EVALUATED (all rules in scope):
✗ NOT FIRED: R_ER_ALERTNESS  (Altered alertness)
✗ NOT FIRED: R_ER_BREATHING  (Breathing distress)
✗ NOT FIRED: R_ER_DEHYDRATION_SEVERE  (Severe dehydration)
✗ NOT FIRED: R_ER_NO_FLUID_NO_URINE  (No fluid intake and no urine output)
✗ NOT FIRED: R_CPG_SEIZURE  (Febrile seizure)
✗ NOT FIRED: R_CPG_INFANT_UNDER_3MO_

---
## Scenario 5 — Local Context (Extended)

**4-year-old**, 102°F, responsive, drinking, voided today.
Parent mentions: *“I think it is just what is going around school.”*

Demonstrates **local context isolation**:
- `local_outbreak_context` is captured and shown in the state
- It is labelled as a **probabilistic prior only** — it never influences any PyDatalog rule
- The symbolic decision (`HOME_MANAGEMENT`) is identical to what it would be without the mention
- The baseline may treat this as a reassuring modifier and under-weight severity in other scenarios

**Expected:** `HOME_MANAGEMENT` with `local_outbreak_context` in state but **no change to disposition**.

In [ ]:

s5 = run_scenario(
    title='Scenario 5 — Local Context (4-year-old, viral context captured but irrelevant)',
    expected='HOME_MANAGEMENT',
    turns=[
        'My 4-year-old has a fever of 102 F. I think it is just what is going around school.',
        'She is alert and talks to me fine. No breathing issues. She is drinking some water.',
        'She peed today.',
    ]
)


---
## Full Harness Verification

In [ ]:
from pathlib import Path
from caretrace.evaluation.harness import run_file

exit_code = run_file(Path('caretrace/evaluation/scenarios.csv'))
print(f'Harness exit code: {exit_code} (0 = all pass)')